# Лабораторная 1: декодерный трансформер для предсказания следующего токена

## Что нужно знать до старта
Перед началом лабораторной полезно открыть:
- [../README.md](./README.md)
- [guides/00_autoregression_prerequisites.md](./guides/00_autoregression_prerequisites.md)
- [guides/01_decoder_only_toy_walkthrough.md](./guides/01_decoder_only_toy_walkthrough.md)
- [../theory/theory.md](../theory/theory.md)

Это первая лабораторная блока `04-Autoregression` и Шаг 7 общего курса.


## Выбор среды выполнения

Рекомендуемый стартовый режим: `auto`.

Если нужна полностью воспроизводимая проверка на центральном процессоре, выберите `local-cpu` и выполните `Restart & Run All`.


In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/04-Autoregression/lab/requirements.txt"


def _detect_notebook_platform():
    """Определяет тип среды выполнения текущей тетради.

    Аргументы:
      Нет.

    Возвращает:
      Строка из множества `{'local', 'colab', 'kaggle'}`.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    """Проверяет, похож ли путь на корень учебного репозитория.

    Аргументы:
      path: Проверяемый путь.

    Возвращает:
      `True`, если обнаружены ключевые признаки корня репозитория.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    """Возвращает стандартный путь клонирования для облачной платформы.

    Аргументы:
      platform: Имя платформы (`'colab'` или `'kaggle'`).

    Возвращает:
      Абсолютный путь каталога репозитория.

    Исключения:
      ValueError: Если передано неподдерживаемое имя платформы.
    """
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    """Проверяет, остался ли в настройке шаблонный URL репозитория.

    Аргументы:
      repo_url: Проверяемый URL репозитория.

    Возвращает:
      `True`, если URL имеет вид шаблона-заглушки.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    """Ищет корень курса, поднимаясь от текущего каталога вверх.

    Аргументы:
      Нет.

    Возвращает:
      Объект `Path` корня репозитория или `None`, если путь не найден.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    """Обеспечивает доступность модуля `course_runtime` для текущей среды.

    Аргументы:
      runtime_mode: Режим запуска тетради.
      repo_url: URL репозитория курса для облачной автозагрузки.

    Возвращает:
      `None`.

    Исключения:
      ModuleNotFoundError: Если локальный запуск выполнен вне корректного корня репозитория.
      RuntimeError: Если в облаке отсутствует валидный URL репозитория или каталог повреждён.
      subprocess.CalledProcessError: Если команда `git clone` завершается с ошибкой.
    """
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


ModuleNotFoundError: Не удалось импортировать course_runtime.py. Для локального запуска открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта.

## Интуиция задачи без формул

Модель предсказывает следующий токен, опираясь только на прошлые позиции. Для этого в блоке внимания используется причинная маска (causal mask).


## Как проходить работу

Маршрут строго фиксирован: данные → маска → блок декодера → обучение → генерация → диагностика внимания.


## Контракт данных

В этой лабораторной используются последовательности фиксированной максимальной длины.

- Вход `X`: идентификаторы токенов без последнего шага, форма `(N, T)`.
- Цель `Y`: те же последовательности, сдвинутые на один шаг влево, форма `(N, T)`.
- Дополнение пустыми позициями задаётся токеном `PAD = 0`.

Разбиение на обучение, проверку и тест выполняется строго индексами без перемешивания.


## Таблица ключевых форм

| Сущность | Смысл | Форма |
|---|---|---|
| `X` | входные токены | `(N, T)` |
| `Y` | цели следующего токена | `(N, T)` |
| `mask` | причинная маска | `(T, T)` |
| `attention_scores` | веса внимания | `(N, H, T, T)` |
| `y_pred` | вероятности токенов | `(N, T, V)` |


## Ручной пример

Пусть последовательность имеет вид:

```text
<BOS> A B C <EOS> <PAD> <PAD>
```

Тогда пара для обучения следующему токену строится так:

```text
X: <BOS> A B C <EOS> <PAD>
Y: A B C <EOS> <PAD> <PAD>
```

То есть на каждом шаге модель должна восстановить токен, стоящий на одну позицию правее.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 13
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
TOKEN_A = 3
TOKEN_B = 4
TOKEN_C = 5
TOKEN_D = 6

ID_TO_TOKEN = {
    PAD_ID: '<PAD>',
    BOS_ID: '<BOS>',
    EOS_ID: '<EOS>',
    TOKEN_A: 'A',
    TOKEN_B: 'B',
    TOKEN_C: 'C',
    TOKEN_D: 'D',
}
VOCAB_SIZE = len(ID_TO_TOKEN)
MAX_SEQ_LEN = 18
PAYLOAD_LEN = 9
CHECK_NEW_TOKENS = 4
TRAIN_SAMPLES = 4096
VAL_SAMPLES = 512
TEST_SAMPLES = 512
TOTAL_SAMPLES = TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES

EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
BATCH_SIZE = 64
EPOCHS = 16

plt.style.use('default')
keras.utils.set_random_seed(SEED)


## TODO 1: подготовка детерминированных данных


In [ ]:
START_PAIRS = np.array(
    [
        [TOKEN_A, TOKEN_B],
        [TOKEN_B, TOKEN_C],
        [TOKEN_C, TOKEN_A],
        [TOKEN_D, TOKEN_B],
        [TOKEN_A, TOKEN_D],
        [TOKEN_D, TOKEN_A],
    ],
    dtype=np.int32,
)


def next_payload_token(prev_prev, prev):
    """Возвращает следующий токен по детерминированному правилу второго порядка.

    Аргументы:
      prev_prev: Предыдущий токен с лагом 2.
      prev: Предыдущий токен с лагом 1.

    Возвращает:
      Идентификатор следующего токена.

    Исключения:
      ValueError: Если токен не принадлежит учебному словарю полезных токенов.
    """
    valid_tokens = {TOKEN_A, TOKEN_B, TOKEN_C, TOKEN_D}
    if prev_prev not in valid_tokens or prev not in valid_tokens:
        raise ValueError('Ожидаются только полезные токены A/B/C/D.')

    map_if_ac = {
        TOKEN_A: TOKEN_B,
        TOKEN_B: TOKEN_C,
        TOKEN_C: TOKEN_A,
        TOKEN_D: TOKEN_B,
    }
    map_if_bd = {
        TOKEN_A: TOKEN_C,
        TOKEN_B: TOKEN_A,
        TOKEN_C: TOKEN_D,
        TOKEN_D: TOKEN_A,
    }
    return map_if_ac[prev] if prev_prev in (TOKEN_A, TOKEN_C) else map_if_bd[prev]


def build_synthetic_corpus(num_samples, max_seq_len=MAX_SEQ_LEN, seed=SEED):
    """Создаёт детерминированный корпус для next-token задачи.

    Следующий токен зависит от двух предыдущих токенов полезной части,
    поэтому задача не сводится к одношаговому соответствию.

    Аргументы:
      num_samples: Число последовательностей в корпусе.
      max_seq_len: Максимальная длина последовательности с учётом дополнения.
      seed: Зерно случайности для воспроизводимости.

    Возвращает:
      Кортеж `(X, Y)`, где обе матрицы имеют форму `(num_samples, max_seq_len - 1)`.

    Исключения:
      ValueError: Если `max_seq_len` недостаточен.
    """
    if max_seq_len < PAYLOAD_LEN + 2:
        raise ValueError('max_seq_len должен быть не меньше PAYLOAD_LEN + 2.')

    # Устанавливаем seed для воспроизводимости
    np.random.seed(seed)
    
    # Инициализируем массив данных с PAD_ID
    data = np.full((num_samples, max_seq_len), PAD_ID, dtype=np.int32)
    
    # Для каждого образца генерируем последовательность
    for i in range(num_samples):
        # Случайно выбираем стартовую пару из START_PAIRS
        start_pair_idx = np.random.randint(0, len(START_PAIRS))
        start_pair = START_PAIRS[start_pair_idx].copy()
        
        # Помещаем стартовую пару в начало последовательности
        data[i, 0] = start_pair[0]
        data[i, 1] = start_pair[1]
        
        # Генерируем полезную нагрузку длины PAYLOAD_LEN
        # Начинаем с последних двух токенов
        prev_prev = start_pair[0]
        prev = start_pair[1]
        
        # Заполняем последовательность начиная с позиции 2
        for j in range(2, PAYLOAD_LEN + 2):
            next_token = next_payload_token(prev_prev, prev)
            data[i, j] = next_token
            prev_prev, prev = prev, next_token
    
    # Создаем X и Y сдвигом
    X = data[:, :-1].copy()  # форма (num_samples, max_seq_len - 1)
    Y = data[:, 1:].copy()   # форма (num_samples, max_seq_len - 1)
    
    # Зануляем первые две целевые позиции (у них нет полного контекста)
    Y[:, 0] = PAD_ID  # нет контекста длины 2
    Y[:, 1] = PAD_ID  # есть только 1 токен контекста
    
    return X, Y


# Генерируем полный корпус
X_all, y_all = build_synthetic_corpus(
    num_samples=NUM_SAMPLES, 
    max_seq_len=MAX_SEQ_LEN, 
    seed=SEED
)

# Фиксированное индексное разбиение без перемешивания
# 70% train, 15% val, 15% test
train_size = int(0.7 * NUM_SAMPLES)
val_size = int(0.15 * NUM_SAMPLES)

# Детерминированное разбиение по порядку (без shuffle)
X_train = X_all[:train_size]
y_train = y_all[:train_size]

X_val = X_all[train_size:train_size + val_size]
y_val = y_all[train_size:train_size + val_size]

X_test = X_all[train_size + val_size:]
y_test = y_all[train_size + val_size:]

print(f"Train size: {len(X_train)}")
print(f"Val size: {len(X_val)}")
print(f"Test size: {len(X_test)}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Мини-проверка после TODO 1
assert X_train.shape == (TRAIN_SAMPLES, MAX_SEQ_LEN - 1)
assert y_train.shape == (TRAIN_SAMPLES, MAX_SEQ_LEN - 1)
assert X_val.shape == (VAL_SAMPLES, MAX_SEQ_LEN - 1)
assert X_test.shape == (TEST_SAMPLES, MAX_SEQ_LEN - 1)

x_a, y_a = build_synthetic_corpus(8, seed=SEED + 101)
x_b, y_b = build_synthetic_corpus(8, seed=SEED + 101)
assert np.array_equal(x_a, x_b), 'Данные не воспроизводятся при одинаковом seed.'
assert np.array_equal(y_a, y_b), 'Цели не воспроизводятся при одинаковом seed.'

# Проверка, что правило действительно второго порядка.
assert next_payload_token(TOKEN_A, TOKEN_B) != next_payload_token(TOKEN_D, TOKEN_B), (
    'Переход должен зависеть от двух предыдущих токенов.'
)
assert np.all(y_a[:, :2] == PAD_ID), (
    'Первые две целевые позиции должны быть замаскированы как PAD_ID.'
)

print('Формы и детерминизм после TODO 1 проверены.')

## TODO 2: причинная маска и её проверка


In [ ]:
def build_causal_mask(seq_len):
    """Строит нижнетреугольную причинную маску.

    Аргументы:
      seq_len: Длина последовательности.

    Возвращает:
      Булев тензор формы `(seq_len, seq_len)`.

    Исключения:
      tf.errors.InvalidArgumentError: Если `seq_len` не является положительным.
    """
    # Проверка что seq_len положительный
    tf.debugging.assert_positive(seq_len)
    
    # Создаем нижнетреугольную маску (включая диагональ)
    # True означает "можно обратить внимание", False - "нельзя"
    mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    
    # Преобразуем в булевый тензор
    mask = tf.cast(mask, tf.bool)
    
    return mask


def assert_lower_triangular(mask):
    """Проверяет, что маска имеет нижнетреугольную структуру.

    Аргументы:
      mask: Булева матрица формы `(T, T)`.

    Исключения:
      AssertionError: Если верхний треугольник содержит разрешающие значения.
    """
    array_mask = np.asarray(mask).astype(np.int32)
    upper = np.triu(array_mask, k=1)
    assert upper.sum() == 0, 'Верхний треугольник должен быть полностью закрыт.'


# Создаем маску длины 8
mask = build_causal_mask(8)

# Проверяем форму
print(f"Маска формы: {mask.shape}")
assert mask.shape == (8, 8), "Форма маски должна быть (8, 8)"

# Проверяем нижнетреугольность
assert_lower_triangular(mask)

# Визуализируем маску для наглядности
print("\nВизуализация causal mask (True = можно attention):")
mask_numpy = mask.numpy()
for i in range(8):
    row = ['✓' if mask_numpy[i, j] else '✗' for j in range(8)]
    print(f"Позиция {i}: " + ' '.join(row))

# Дополнительная проверка: токен на позиции i может видеть токены <= i
for i in range(8):
    for j in range(8):
        if j <= i:
            assert mask_numpy[i, j] == True, f"Токен {i} должен видеть {j}"
        else:
            assert mask_numpy[i, j] == False, f"Токен {i} НЕ должен видеть {j}"

print("\n✅ Все проверки causal mask пройдены!")

## TODO 3: декодерный блок и метрики


In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    """Складывает векторы токенов и позиций в едином представлении."""

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        """Инициализирует таблицы токенных и позиционных векторов."""
        super().__init__(**kwargs)
        if embed_dim < 1:
            raise ValueError('embed_dim должен быть положительным.')
        self.maxlen = maxlen
        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        """Вычисляет сумму токенных и позиционных векторов."""
        # Получаем batch_size и длину последовательности
        seq_len = tf.shape(inputs)[1]
        
        # Создаем позиции от 0 до seq_len-1
        positions = tf.range(start=0, limit=seq_len, delta=1)
        
        # Получаем эмбеддинги для токенов и позиций
        token_embeds = self.token_embedding(inputs)
        position_embeds = self.position_embedding(positions)
        
        # Складываем эмбеддинги
        return token_embeds + position_embeds

    def compute_mask(self, inputs, mask=None):
        """Переадресует маску непустых токенов из слоя токенных векторов."""
        return self.token_embedding.compute_mask(inputs)


class CausalDecoderBlock(layers.Layer):
    """Минимальный декодерный блок с причинной маской."""

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        """Создаёт внутренние слои блока."""
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')

        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ],
            name='positionwise_ffn',
        )
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(rate)
        self.dropout_2 = layers.Dropout(rate)

    def call(self, inputs, padding_mask=None, training=None, return_attention_scores=False):
        """Прогоняет вход через внимание и позиционно-независимую сеть."""
        seq_len = tf.shape(inputs)[1]
        
        # Создаем причинную маску
        causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        causal_mask = tf.cast(causal_mask, tf.bool)
        
        # Комбинируем causal_mask с padding_mask если необходимо
        if padding_mask is not None:
            # padding_mask имеет форму (batch, seq_len)
            # Добавляем измерение для attention: (batch, 1, 1, seq_len)
            padding_mask = padding_mask[:, tf.newaxis, tf.newaxis, :]
            # Объединяем маски: позиция доступна только если обе маски разрешают
            combined_mask = causal_mask[tf.newaxis, tf.newaxis, :, :] & padding_mask
        else:
            combined_mask = causal_mask[tf.newaxis, tf.newaxis, :, :]
        
        # Self-attention с маской
        attn_output, attn_scores = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=combined_mask,
            return_attention_scores=True,
            training=training,
        )
        
        # Residual connection + normalization
        attn_output = self.dropout_1(attn_output, training=training)
        out_1 = self.norm_1(inputs + attn_output)
        
        # Position-wise FFN
        ffn_output = self.ffn(out_1, training=training)
        ffn_output = self.dropout_2(ffn_output, training=training)
        
        # Residual connection + normalization
        out_2 = self.norm_2(out_1 + ffn_output)
        
        if return_attention_scores:
            return out_2, attn_scores
        return out_2

    def compute_mask(self, inputs, mask=None):
        """Отключает автоматическое пробрасывание маски в выход блока."""
        return None


def masked_sparse_crossentropy(y_true, y_pred):
    """Считает перекрёстную энтропию только по непустым позициям."""
    per_token = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    masked = per_token * mask
    return tf.reduce_sum(masked) / tf.reduce_sum(mask)


def masked_token_accuracy(y_true, y_pred):
    """Считает токенную точность только по непустым позициям."""
    predicted = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    correct = tf.cast(tf.equal(y_true, predicted), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(correct * mask) / tf.reduce_sum(mask)


def perplexity_from_loss(loss_value):
    """Преобразует среднюю перекрёстную энтропию в перплексию."""
    return float(np.exp(loss_value))

## TODO 4: сборка и обучение модели


In [ ]:
# TODO 4.1: Соберите модель с TokenAndPositionEmbedding и CausalDecoderBlock.
# TODO 4.2: Скомпилируйте модель с masked_sparse_crossentropy и masked_token_accuracy.
# TODO 4.3: Обучите модель на train и validation без перемешивания.

# Определяем параметры модели
VOCAB_SIZE = 6  # TOKEN_A, B, C, D, PAD_ID, UNK (или другие)
EMBED_DIM = 32
NUM_HEADS = 4
FF_DIM = 64
NUM_LAYERS = 2
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
EPOCHS = 50
BATCH_SIZE = 32

# Строим модель
inputs = layers.Input(shape=(MAX_SEQ_LEN - 1,), name='input_tokens')

# Слой эмбеддингов (токен + позиция)
x = TokenAndPositionEmbedding(
    maxlen=MAX_SEQ_LEN - 1,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    name='token_position_embedding'
)(inputs)

# Добавляем несколько декодерных блоков
for i in range(NUM_LAYERS):
    x = CausalDecoderBlock(
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM,
        rate=DROPOUT_RATE,
        name=f'causal_decoder_block_{i}'
    )(x, padding_mask=None)  # padding_mask будет добавлен позже при необходимости

# Выходной слой для предсказания следующего токена
outputs = layers.Dense(VOCAB_SIZE, activation='softmax', name='token_predictions')(x)

# Создаем модель
model = keras.Model(inputs=inputs, outputs=outputs, name='decoder_only_transformer')

# Компилируем модель с masked_sparse_crossentropy и masked_token_accuracy
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=masked_sparse_crossentropy,
    metrics=[masked_token_accuracy]
)

# Выводим архитектуру модели
print("Архитектура модели:")
model.summary()

# Обучаем модель на train и validation без перемешивания
history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    shuffle=False,  # Без перемешивания как указано в задании
    verbose=1
)

print("\n✅ Модель успешно собрана и обучена!")

# Визуализация процесса обучения
import matplotlib.pyplot as plt

# График потерь
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (Crossentropy)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# График точности
plt.subplot(1, 2, 2)
plt.plot(history.history['masked_token_accuracy'], label='Train Accuracy')
plt.plot(history.history['val_masked_token_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Оценка на тестовых данных
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_perplexity = perplexity_from_loss(test_loss)

print(f"\n📊 Результаты на тестовой выборке:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Perplexity: {test_perplexity:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

# Проверка критериев успеха
if test_accuracy >= 0.97 and test_perplexity <= 1.30:
    print("\n✅ Критерии успеха достигнуты!")
    print(f"   Accuracy {test_accuracy:.4f} >= 0.97")
    print(f"   Perplexity {test_perplexity:.4f} <= 1.30")
else:
    print("\n⚠️ Критерии успеха НЕ достигнуты:")
    if test_accuracy < 0.97:
        print(f"   Accuracy {test_accuracy:.4f} < 0.97")
    if test_perplexity > 1.30:
        print(f"   Perplexity {test_perplexity:.4f} > 1.30")

In [ ]:
# Графики после TODO 4
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='обучение')
plt.plot(history.history['val_loss'], label='проверка')
plt.title('Перекрёстная энтропия по эпохам')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
train_ppl = [perplexity_from_loss(v) for v in history.history['loss']]
val_ppl = [perplexity_from_loss(v) for v in history.history['val_loss']]
plt.plot(train_ppl, label='обучение')
plt.plot(val_ppl, label='проверка')
plt.title('Перплексия по эпохам')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()


In [ ]:
# Контроль порогов после TODO 4
test_loss, test_token_acc = model.evaluate(X_test, y_test, verbose=0)
test_perplexity = perplexity_from_loss(test_loss)
print(f'test_loss            : {test_loss:.4f}')
print(f'test_token_accuracy  : {test_token_acc:.4f}')
print(f'test_perplexity      : {test_perplexity:.4f}')
assert test_token_acc >= 0.97, 'Токенная точность ниже порога 0.97.'
assert test_perplexity <= 1.30, 'Перплексия выше порога 1.30.'


## TODO 5: детерминированная генерация


In [ ]:
def generate_greedy(model, prompt_ids, max_new_tokens, max_seq_len=MAX_SEQ_LEN - 1):
    """Генерирует продолжение в детерминированном режиме argmax.

    Аргументы:
      model: Обученная модель предсказания следующего токена.
      prompt_ids: Начальная последовательность идентификаторов токенов.
      max_new_tokens: Максимум новых шагов генерации.
      max_seq_len: Рабочая длина входа модели.

    Возвращает:
      Список идентификаторов с учётом исходной подсказки и сгенерированных токенов.

    Исключения:
      ValueError: Если подсказка пуста.
    """
    if len(prompt_ids) == 0:
        raise ValueError('Подсказка не может быть пустой.')
    
    # Копируем начальную последовательность
    generated = list(prompt_ids)
    
    for _ in range(max_new_tokens):
        # Берем последние max_seq_len токенов (или меньше для начала)
        current_seq = generated[-max_seq_len:] if len(generated) > max_seq_len else generated
        
        # Преобразуем в тензор с формой (1, seq_len)
        input_tensor = tf.expand_dims(tf.convert_to_tensor(current_seq), axis=0)
        
        # Получаем предсказания модели
        predictions = model(input_tensor, training=False)
        
        # Берем предсказание для последней позиции
        next_token_logits = predictions[0, -1, :]
        
        # Выбираем токен с максимальной вероятностью (argmax)
        next_token = tf.argmax(next_token_logits).numpy()
        
        # Добавляем в последовательность
        generated.append(next_token)
    
    return generated


def expected_payload_steps(prompt_ids, steps):
    """Строит эталонное продолжение по детерминированному правилу второго порядка.

    Аргументы:
      prompt_ids: Начальная последовательность токенов вида `[BOS, t1, t2, ...]`.
      steps: Сколько новых токенов нужно построить.

    Возвращает:
      Список из `steps` токенов эталонного продолжения.

    Исключения:
      ValueError: Если полезная часть подсказки короче двух токенов.
    """
    payload = [token for token in prompt_ids if token in (TOKEN_A, TOKEN_B, TOKEN_C, TOKEN_D)]
    if len(payload) < 2:
        raise ValueError('Для эталонного продолжения нужно минимум два полезных токена.')

    generated = []
    history = list(payload)
    for _ in range(steps):
        next_token = next_payload_token(history[-2], history[-1])
        generated.append(next_token)
        history.append(next_token)
    return generated


# TODO 5.2: Запустите 20 фиксированных подсказок и проверьте порог 18/20
# по ПОЛНОМУ контролируемому продолжению длины CHECK_NEW_TOKENS.

# Параметры проверки
CHECK_NEW_TOKENS = 10  # Количество генерируемых токенов для проверки
NUM_PROMPTS = 20  # Количество тестовых подсказок
SUCCESS_THRESHOLD = 18  # Нужно минимум 18 успешных из 20

# Создаем фиксированный набор подсказок из тестовой выборки
np.random.seed(SEED)
test_indices = np.random.choice(len(X_test), size=NUM_PROMPTS, replace=False)

success_count = 0
results = []

print(f"Проверка генерации на {NUM_PROMPTS} подсказках (нужно минимум {SUCCESS_THRESHOLD} успешных):")
print("=" * 70)

for i, idx in enumerate(test_indices):
    # Берем подсказку из тестовой выборки (первые 5 токенов)
    prompt = X_test[idx, :5].tolist()
    # Убираем PAD_ID из начала если есть
    prompt = [token for token in prompt if token != PAD_ID]
    
    if len(prompt) < 2:
        # Если подсказка слишком короткая, пропускаем
        continue
    
    # Генерируем продолжение моделью
    generated = generate_greedy(model, prompt, max_new_tokens=CHECK_NEW_TOKENS)
    
    # Получаем эталонное продолжение
    expected = expected_payload_steps(prompt, CHECK_NEW_TOKENS)
    
    # Сравниваем сгенерированные токены с эталоном
    generated_tokens = generated[len(prompt):]  # Только новые токены
    is_correct = generated_tokens == expected
    
    # Подсчитываем количество совпадений
    matches = sum(is_correct)
    is_full_match = matches == CHECK_NEW_TOKENS
    
    if is_full_match:
        success_count += 1
        status = "✓ УСПЕХ"
    else:
        status = f"✗ НЕУДАЧА (совпадений: {matches}/{CHECK_NEW_TOKENS})"
    
    results.append({
        'prompt': prompt,
        'expected': expected,
        'generated': generated_tokens,
        'matches': matches,
        'full_match': is_full_match
    })
    
    print(f"{i+1:2d}. {status}")
    if not is_full_match:
        print(f"     Подсказка: {prompt}")
        print(f"     Ожидалось: {expected}")
        print(f"     Сгенерировано: {generated_tokens}")

print("=" * 70)
print(f"\nРезультат: {success_count}/{NUM_PROMPTS} успешных продолжений")

# Проверяем критерий успеха
if success_count >= SUCCESS_THRESHOLD:
    print(f"\n✅ КРИТЕРИЙ УСПЕХА ДОСТИГНУТ: {success_count} >= {SUCCESS_THRESHOLD}")
else:
    print(f"\n❌ КРИТЕРИЙ УСПЕХА НЕ ДОСТИГНУТ: {success_count} < {SUCCESS_THRESHOLD}")

# Дополнительная проверка среднего совпадения
mean_match_ratio = np.mean([r['matches'] for r in results]) / CHECK_NEW_TOKENS
print(f"Средняя доля совпадений: {mean_match_ratio:.2%}")

# Визуализация одного примера
print("\n" + "=" * 70)
print("Пример успешной генерации:")
success_examples = [r for r in results if r['full_match']]
if success_examples:
    example = success_examples[0]
    print(f"Подсказка: {example['prompt']}")
    print(f"Ожидаемое продолжение: {example['expected']}")
    print(f"Сгенерированное продолжение: {example['generated']}")
else:
    print("Нет успешных примеров для отображения")

## TODO 6: диагностика внимания


In [ ]:
# TODO 6.1: Соберите tracing-path, который возвращает attention_scores из CausalDecoderBlock.
# TODO 6.2: Усредните attention_scores по головам и обрежьте карту до непустых токенов.
# TODO 6.3: Проверьте, что отношение массы внимания в будущем меньше 1e-4.

# Создаем модель для диагностики внимания
def build_attention_tracing_model(base_model, decoder_block_index=-1):
    """Создает модель, которая возвращает attention_scores из указанного блока."""
    # Находим последний CausalDecoderBlock
    decoder_blocks = [layer for layer in base_model.layers if isinstance(layer, CausalDecoderBlock)]
    if not decoder_blocks:
        raise ValueError("В модели нет CausalDecoderBlock слоев")
    
    target_block = decoder_blocks[decoder_block_index]
    
    # Строим модель, которая возвращает attention_scores
    input_layer = base_model.input
    x = input_layer
    
    # Пропускаем через все слои до target_block
    reached_target = False
    for layer in base_model.layers[1:]:  # Пропускаем input layer
        if layer == target_block:
            # Для целевого блока вызываем с return_attention_scores=True
            x, attention_scores = layer(x, return_attention_scores=True)
            reached_target = True
        elif reached_target:
            # Для последующих слоев без attention_scores
            x = layer(x)
        else:
            # Для слоев до целевого блока
            x = layer(x)
    
    if not reached_target:
        raise ValueError("Целевой блок не найден в модели")
    
    tracing_model = keras.Model(inputs=input_layer, outputs=attention_scores, name='attention_tracer')
    return tracing_model


# Строим tracing модель
attention_tracer = build_attention_tracing_model(model, decoder_block_index=-1)  # Берем последний блок
print(f"Модель для трассировки внимания создана: {attention_tracer.name}")

# Берем один батч из тестовой выборки для диагностики
sample_idx = 0
sample_input = X_test[sample_idx:sample_idx+1]  # Форма (1, seq_len)

# Получаем attention_scores
attention_scores = attention_tracer(sample_input, training=False)
print(f"Attention scores shape: {attention_scores.shape}")  # (batch, num_heads, query_len, key_len)

# TODO 6.2: Усредните attention_scores по головам и обрежьте карту до непустых токенов.
# Усредняем по головам
avg_attention = tf.reduce_mean(attention_scores, axis=1)  # (batch, query_len, key_len)
avg_attention = avg_attention[0].numpy()  # Убираем batch dimension

# Обрезаем до непустых токенов (убираем PAD_ID)
# Находим реальные токены в input
input_tokens = X_test[sample_idx]
non_pad_mask = input_tokens != PAD_ID
non_pad_indices = np.where(non_pad_mask)[0]
non_pad_len = len(non_pad_indices)

if non_pad_len > 0:
    # Обрезаем attention карту до непустых позиций
    trimmed_attention = avg_attention[:non_pad_len, :non_pad_len]
    print(f"Обрезанная attention карта формы: {trimmed_attention.shape}")
else:
    trimmed_attention = avg_attention
    print("Внимание: последовательность не содержит непустых токенов")

# Визуализация attention карты
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.imshow(trimmed_attention, cmap='Blues', aspect='auto')
plt.colorbar(label='Attention weight')
plt.xlabel('Key positions (context)')
plt.ylabel('Query positions (current)')
plt.title(f'Attention Map (averaged over heads)\nSequence length: {non_pad_len}')
plt.grid(False)

# Добавляем разделительную линию между прошлым и будущим
# Диагональ - это текущий токен, выше диагонали - будущее
plt.plot([-0.5, non_pad_len-0.5], [-0.5, non_pad_len-0.5], 'r--', linewidth=2, label='causal boundary')
plt.legend()
plt.tight_layout()
plt.show()

# Вычисляем массу внимания к будущим токенам (выше диагонали)
future_mask = np.triu(np.ones_like(trimmed_attention), k=1).astype(bool)
future_attention_mass = trimmed_attention[future_mask].sum()
total_attention_mass = trimmed_attention.sum()

future_ratio = future_attention_mass / total_attention_mass if total_attention_mass > 0 else 0

print(f"\n📊 Диагностика причинности:")
print(f"  Полная масса внимания: {total_attention_mass:.6f}")
print(f"  Масса внимания к будущему: {future_attention_mass:.6f}")
print(f"  Отношение к будущему: {future_ratio:.2e}")

# TODO 6.3: Проверьте, что отношение массы внимания в будущем меньше 1e-4.
threshold = 1e-4
if future_ratio < threshold:
    print(f"\n✅ УТЕЧЕК НЕТ: отношение {future_ratio:.2e} < {threshold}")
    print("   Causal mask работает корректно, модель не видит будущие токены.")
else:
    print(f"\n❌ ОБНАРУЖЕНЫ УТЕЧКИ: отношение {future_ratio:.2e} >= {threshold}")
    print("   Модель имеет доступ к будущим токенам!")

# Дополнительная проверка: максимальный вес в будущем
max_future_attention = trimmed_attention[future_mask].max() if future_mask.any() else 0
print(f"  Максимальный вес attention в будущем: {max_future_attention:.6f}")

# Проверка на нескольких примерах
print("\n" + "=" * 70)
print("Проверка на 10 случайных примерах из тестовой выборки:")
threshold = 1e-4
violations = 0

for idx in range(min(10, len(X_test))):
    sample_input = X_test[idx:idx+1]
    attention_scores = attention_tracer(sample_input, training=False)
    avg_attention = tf.reduce_mean(attention_scores, axis=1)[0].numpy()
    
    # Находим непустые токены
    input_tokens = X_test[idx]
    non_pad_mask = input_tokens != PAD_ID
    non_pad_len = np.sum(non_pad_mask)
    
    if non_pad_len > 1:
        trimmed = avg_attention[:non_pad_len, :non_pad_len]
        future_mask = np.triu(np.ones_like(trimmed), k=1).astype(bool)
        future_mass = trimmed[future_mask].sum()
        total_mass = trimmed.sum()
        ratio = future_mass / total_mass if total_mass > 0 else 0
        
        status = "✓" if ratio < threshold else "✗"
        if ratio >= threshold:
            violations += 1
        print(f"  {idx+1:2d}. {status} Отношение к будущему: {ratio:.2e}")
    else:
        print(f"  {idx+1:2d}. - Слишком короткая последовательность")

print("=" * 70)
if violations == 0:
    print(f"\n✅ Все проверки пройдены! Нет утечек в будущее.")
else:
    print(f"\n⚠️ Найдено {violations} нарушений причинности!")

# Итоговый вывод
print("\n" + "=" * 70)
print("ИТОГ ДИАГНОСТИКИ ВНИМАНИЯ:")
if future_ratio < threshold and violations == 0:
    print("✅ CAUSAL MASK РАБОТАЕТ КОРРЕКТНО")
    print("   Модель не имеет доступа к будущим токенам")
else:
    print("❌ ОБНАРУЖЕНА УТЕЧКА ИНФОРМАЦИИ")
    print("   Проверьте реализацию causal mask")
print("=" * 70)

## Чек-лист перед завершением
1. Все `TODO` закрыты.
2. Воспроизводимость данных подтверждена на первых `k` примерах.
3. Токенная точность и перплексия проходят пороги.
4. Детерминированная генерация выполняет порог `18/20`.
5. Карта внимания не содержит доступа к будущим позициям.
